# Week 4 Activity: Delay, Filters, Reverb

Complete this activity as part of your participation grade. Pending length of the lecture, you will have time in class to work. Everything you need to complete this activity can be found in this week's (or a previous week's) lecture code. For this activity, you may want to consult your Audio Tech I notes (or the Computer Music Tutorial)

In [5]:
import numpy as np
from scipy.io.wavfile import read
from scipy.signal import sawtooth, square, butter, filtfilt
import matplotlib.pyplot as plt
from IPython.display import Audio, Image
fs = 44100

## Delay
1) Create a function that will modify a passed signal by adding a delay of $m$ milliseconds to the signal.

In [7]:

def delay(signal, offset, fs):
    copy = signal.copy()
    pad = np.zeros(int(fs/1000*offset))
    orig = np.concatenate([signal,pad])
    delayed = np.concatenate([pad,copy])
    return orig+delayed

2) Modify the function such that you can optionally scale the amplitude of the delayed signal 

In [8]:

def scaledDelay(signal, offset, fs, amplitude=0.5):
    copy = signal.copy()
    pad = np.zeros(int(fs/1000*offset))
    orig = np.concatenate([signal,pad])
    delayed = np.concatenate([pad,copy])
    delayed *= amplitude
    return orig+delayed


3) Modify the function so that the user can specify the number of delays to add to the original signal (and optionally, scale each subsequent delay amplitude by a factor of $1/n$

In [9]:

def multipleDelays(signal, offset, fs, amplitude=0.5, repeats=1, scale=False):
    copy = signal.copy()
    delay = np.concatenate([copy, np.zeros(int(fs/1000*offset*repeats))])
    for i in range(repeats+1):
        #create some silence
        pad_beg = np.zeros(int(fs/1000*offset*i))
        pad_end = np.zeros(int(fs/1000*offset*(repeats-i)))
        # add to the original starting from first delay
        currDelay = np.concatenate([pad_beg,copy,pad_end])
        if scale:
            currDelay *= 1/(i+1)
        delay += currDelay
    return delay



## Filters
1. Create a function that will apply an feedforward comb filter by computing the delay length based off a given resonant frequency in Hz.

In [12]:
def ffComb(x, fs, fRes, g=0.8):
    x = np.asarray(x, dtype=float)
    D = int(np.round(fs / float(fRes))) # converts resonant frequency to delay in samples
    y = np.zeros_like(x)
    for n in range(len(x)):
        if n >= D:
            y[n] = x[n] + g * x[n - D]
        else:
            y[n] = x[n]
    return y

2. Create a function that will apply an feedback comb filter by computing the delay length based off a given resonant frequency in Hz.

In [ ]:
def fbComb(x, fs, fRes, g=0.8):
    x = np.asarray(x, dtype=float)
    D = int(np.round(fs / float(fRes)))
    y = np.zeros_like(x)
    for n in range(len(x)):
        if n >= D:
            y[n] = x[n] + g * y[n - D]
        else:
            y[n] = x[n]
    return y

4. Use your comb filter functions on a wave from the audio folder. Try applying different resonant frequencies and delay lengths. How are the filter results different?

In [ ]:
fs = 44100
dur = 1
(fs, x) = read('../audio/sine-101.wav')

y = ffComb(x, fs, fRes=5000)
y2 = ffComb(x, fs, fRes=500)
y3 = ffComb(x, fs, fRes=5)

y4 = fbComb(x, fs, fRes=5000)
y5 = fbComb(x, fs, fRes=500)
y6 = fbComb(x, fs, fRes=5)
Audio(y6, rate=fs)

# changing the resonant frequency changes the delay length, which changes the spacing of peaks

3. Create a function that will apply a butterworth filter to a signal with filter type options 'highpass', 'lowpass', 'bandpass', and 'bandstop'.

In [ ]:
def butterFilter(x, fs, cutoff, order, ftype):
    nyq = fs / 2
    # normalize cutoff
    if ftype == 'lowpass' or ftype == 'highpass':
        Wn = cutoff / nyq
    else:  # bandpass or bandstop
        Wn = (cutoff[0] / nyq, cutoff[1] / nyq)
    b, a = signal.butter(order, Wn, btype=ftype)
    y = signal.lfilter(b, a, x)
    return y

## Reverb/Convolution

1. Create a function that will apply a simple moving average filter by convolving the filter kernel and an incoming signal.

In [ ]:

def movingAverage(x, k):
    h = np.ones(k) / k
    y = np.convolve(x, h, mode='same')
    return y

2. Apply your filter to a noise signal. What is the effect? What happens if you increase or decrease the kernel size?

In [ ]:
(fs, x) = read('../audio/ocean.wav')
y = movingAverage(x, 5)
y2 = movingAverage(x, 100)
y3 = movingAverage(x, 500)

Audio(y2, rate=fs)

# when you increase the kernel size, it makes the sound deeper, heavier lowpass filter is applied

3. Create a function that applies convolution reverb to an input signal given an impulse response (this can be default loaded from the audio files). Use np.convolve to create this function.

In [ ]:
def convReverb(x, h):
    x = np.asarray(x, dtype=float)
    h = np.asarray(h, dtype=float)

    y = np.convolve(x, h)

    return y


4. Create another function that applies convolution reverb to an input signal given an impulse response, but this time do not use np.convolve. You should write the convolution from scratch.

    Challenge yourself to create the most efficient function and time your implementation against np.convolve. 
    While developing and testing your function, do not use real audio files. Start with short signals (e.g., impulses, noise, or short sinusoids). Using long signals with loop-based implementations will result in extremely slow run times.

    **Hint:** in a similar manner to how you may have designed your delay function, recall the functions `numpy.zeros` and `numpy.roll` along with how to manipulate multidimensional numpy arrays (e.g., scalar product, summing columns with `vstack`, etc.) If you plan to try `numpy.roll` DO NOT use it in a loop for convolution with a real audio file! (You'll kill your memory), instead consider the `map` function. You may also want to check out the following function which is similar to numpy.roll but more efficient for this task: `scipy.linalg.circulant`.

    You may wish to visit [here](https://numpy.org/doc/stable/user/basics.broadcasting.html) for review of broadcasting (i.e., form some calculation across index value I and column value C) in numpy

In [42]:

def conv(x, h):
    x = np.asarray(x, dtype=float)
    h = np.asarray(h, dtype=float)
    N = len(x)
    M = len(h)
    y = np.zeros(N + M - 1)
    for i in range(N + M - 1):
        s = 0.0
        kmin = max(0, i - (M - 1))
        kmax = min(i, N - 1)
        for k in range(kmin, kmax + 1):
            s += x[k] * h[i - k]
        y[i] = s
    return y
f = np.array([1,2,3,4])
g = np.array([5,6,7,8,9])
conv(f, g)


array([ 5., 16., 34., 60., 70., 70., 59., 36.])